# FROG: Fine-Tuned WikidataGraphRAG Implementation

## Setup and Dependencies

In [ ]:
# Install required packages
!pip install -q unsloth==2025.5.7
!pip install -q sentence-transformers SPARQLWrapper weaviate-client googletrans-py==4.0.0 rdflib \
           langchain langchain-core langchain-community langchain-huggingface==0.1.2 pydantic nltk dotenv
!pip install -q protobuf==3.20.*

In [ ]:
# Create directory structure
import os

os.makedirs('data/wikidata_ontology', exist_ok=True)
os.makedirs('logs', exist_ok=True)

In [ ]:
# Download Wikidata ontology data and test datasets
!wget -O data/wikidata_ontology/properties.csv https://raw.githubusercontent.com/gansixeneh/FROG-2.0/refs/heads/dataset/dataset/ontology/properties.csv
!wget -O test_data.json https://raw.githubusercontent.com/gansixeneh/FROG-2.0/refs/heads/dataset/dataset/qald_9_plus/qald_9_plus_test_wikidata_converted.json

In [ ]:
# Set up deterministic behavior for reproducibility
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

import torch
import random
import numpy as np

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [ ]:
# # Unsloth imports for optimized model loading
from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported

# # For fine-tuning
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split

import torch
import pandas as pd
import os
import json
import gc
import re
import nltk
import numpy as np
from dotenv import load_dotenv
from copy import deepcopy
from xml.sax.saxutils import escape
from IPython.display import HTML, display
from SPARQLWrapper import SPARQLWrapper, JSON
import requests
# import weaviate
# import weaviate.classes as wvc
from sentence_transformers import SentenceTransformer
from googletrans import Translator
import googletrans
from typing import List, Optional, Dict, Any, Tuple, Union
from pydantic import BaseModel, Field
from transformers import pipeline
from tqdm import tqdm
import matplotlib.pyplot as plt

# langchain imports
from langchain_core.output_parsers import StrOutputParser
from langchain.chains import LLMChain
from langchain.output_parsers import (
    ResponseSchema,
    StructuredOutputParser,
    PydanticOutputParser,
)
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain.output_parsers.prompts import NAIVE_FIX_PROMPT

# Download NLTK data
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from nltk import ngrams

In [ ]:
# Helper functions
def replace_using_dict(original_string, replacements) -> str:
    for old, new in replacements.items():
        original_string = original_string.replace(old, new)
    return original_string

def separate_camel_case(s) -> str:
    separated = re.sub("([a-z])([A-Z])", r"\1 \2", s)
    return separated

def contains_multiple_entities(question) -> bool:
    keywords = ["and", "or", "as well as", "both", "along with", "together with"]
    question = question.lower()
    return any(
        f" {keyword} " in question
        or question.startswith(f"{keyword} ")
        or question.endswith(f" {keyword}")
        for keyword in keywords
    )

def fix_query_spacing(query: str) -> str:
    query = re.sub(r"(select)(\?\w+)", r"\1 \2", query)
    query = re.sub(r"(\w+:\w+)(\?\w+)", r"\1 \2", query)
    return query

In [ ]:
def clear_memory():
    """Aggressively clear GPU memory"""
    import gc
    import torch
    
    # Clear CUDA cache
    torch.cuda.empty_cache()
    
    # Force garbage collection
    gc.collect()
    
    # Add synchronization
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    # Log memory status
    if torch.cuda.is_available():
        print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
        print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

## Wikidata API Implementation


In [ ]:
# Wikidata API
class WikidataAPI:
    def __init__(self, url="https://query.wikidata.org/sparql") -> None:
        self.sparqlwd = SPARQLWrapper(
            url,
            agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.11 (KHTML, like Gecko) Chrome/23.0.1271.64 Safari/537.11",
        )

    def execute_sparql(self, q: str) -> tuple[list[dict], Exception]:
        self.sparqlwd.setQuery(q)
        self.sparqlwd.setReturnFormat(JSON)
        try:
            results = self.sparqlwd.query().convert()
            results_cleaned = []
            for result in results["results"]["bindings"]:
                tmp = dict()
                for header in results["head"]["vars"]:
                    tmp[header] = result[header]["value"]
                results_cleaned.append(tmp)
            return results_cleaned, None
        except Exception as e:
            return [], e

    def execute_sparql_to_df(self, q: str):
        self.sparqlwd.setQuery(q)
        self.sparqlwd.setReturnFormat(JSON)
        results = self.sparqlwd.query().convert()
        df = []
        for result in results["results"]["bindings"]:
            row = {}
            for key, value in result.items():
                row[key] = value["value"]
            df.append(row)
        return pd.DataFrame(df)

    def get_entities(self, entity: str, k: int = 5, lang: str = "en") -> tuple[list[dict], Exception]:
        wikidata_api = "https://www.wikidata.org/w/api.php"
        params = {
            "action": "wbsearchentities",
            "format": "json",
            "search": entity,
            "language": lang,
        }
        try:
            data = requests.get(wikidata_api, params=params)
        except Exception as e:
            return [], e

        json_data = data.json()
        parsed_data = [
            {
                "uri": item["id"],
                "label": item["label"],
                "description": item.get("description", ""),
            }
            for item in json_data["search"][:k]
        ]
        return parsed_data, None

## Verbalization Implementation

In [ ]:
# Modified WikidataVerbalization class with standardized output format
class WikidataVerbalization:
    SENTENCE_TEMPLATE = "{s}'s {p} is {o}"
    MANUAL_MAPPING_DICT = {"_": " "}
    PO_TEMPLATE = """
SELECT distinct ?p ?o ?sLabel ?propLabel ?oLabel
WHERE {{
  BIND(wd:{entity} AS ?s) .
  
  ?s ?p ?o .
  FILTER(?p != wd:P18)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
  ?prop wikibase:directClaim ?p .
}}
"""
    SP_TEMPLATE = """
SELECT ?s ?p ?sLabel ?propLabel ?oLabel
WHERE {{
  BIND(wd:{entity} AS ?o) .
  
  ?s ?p ?o .
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
  ?prop wikibase:directClaim ?p .
}}
"""

    def __init__(
        self,
        model_name="jinaai/jina-embeddings-v3",
        model_kwargs={"trust_remote_code": True},
        query_model_encode_kwargs={},
        passage_model_encode_kwargs={},
    ) -> None:
        self.model_name = model_name
        self.query_model_encode_kwargs = query_model_encode_kwargs
        self.passage_model_encode_kwargs = passage_model_encode_kwargs
        self.model = SentenceTransformer(model_name, **model_kwargs)
        self.model.eval()
        self.api = WikidataAPI()

    def get_po(self, entity: str) -> pd.DataFrame:
        query = self.PO_TEMPLATE.format(entity=entity)
        df = self.api.execute_sparql_to_df(query).drop_duplicates()
        if df.empty:
            return pd.DataFrame(columns=["p", "o", "sLabel", "pLabel", "oLabel"])
        return df

    def get_sp(self, entity: str) -> pd.DataFrame:
        query = self.SP_TEMPLATE.format(entity=entity)
        df = self.api.execute_sparql_to_df(query).drop_duplicates()
        if df.empty:
            return pd.DataFrame(columns=["s", "p", "sLabel", "pLabel", "oLabel"])
        return df

    def get_list_of_candidates(self, entity: str):
        po, sp = self.get_po(entity), self.get_sp(entity)
        candidates = dict()

        # Process predicate-object pairs
        curr_p = None
        for _, (p, o, sLabel, pLabel, oLabel) in po.iterrows():
            label_s = sLabel if sLabel else replace_using_dict(entity.split("/")[-1], self.MANUAL_MAPPING_DICT)
            label_p = pLabel if pLabel else separate_camel_case(p.split("/")[-1])

            if label_p != curr_p:
                curr_p = label_p
                if o.startswith("http"):
                    label_o = oLabel if oLabel else replace_using_dict(o.split("/")[-1], self.MANUAL_MAPPING_DICT)
                else:
                    label_o = o
                candidates[p] = self.SENTENCE_TEMPLATE.format(
                    s=str(label_s), p=str(label_p), o=str(label_o)
                )

        # Process subject-predicate pairs
        curr_p = None
        for _, (s, p, sLabel, pLabel, oLabel) in sp.iterrows():
            label_s = sLabel if sLabel else replace_using_dict(s.split("/")[-1], self.MANUAL_MAPPING_DICT)
            label_p = pLabel if pLabel else separate_camel_case(p.split("/")[-1])
            label_o = oLabel if oLabel else replace_using_dict(entity.split("/")[-1], self.MANUAL_MAPPING_DICT)

            if label_p != curr_p:
                curr_p = label_p
                candidates[p] = self.SENTENCE_TEMPLATE.format(
                    s=str(label_s), p=str(label_p), o=str(label_o)
                )

        return candidates, po, sp

    def run(self, question: str, entity: str) -> tuple[list[dict[str, str]], float]:
        # Get candidate sentences
        list_of_candidates, po, sp = self.get_list_of_candidates(entity)
        cands = list(list_of_candidates.values())
        if not cands:  # Handle empty candidates
            return [], 0.0
            
        # Encode question and candidates
        question_embed = self.model.encode(
            question,
            **self.query_model_encode_kwargs
        )
        passages_embed = self.model.encode(
            cands,
            **self.passage_model_encode_kwargs
        )

        # Find most similar candidate
        similarities = self.model.similarity(question_embed, passages_embed).numpy().flatten()
        similar_index = np.argmax(similarities)
        similar_score = max(similarities)

        # Extract results based on the most similar property
        property_used = list(list_of_candidates.keys())[similar_index]
        result = []
        
        # Add predicate-object pairs
        for _, (p, o, _, pLabel, oLabel) in po[po["p"] == property_used].iterrows():
            result.append({"val": o})
            
        # Add subject-predicate pairs
        for _, (s, p, sLabel, pLabel, _) in sp[sp["p"] == property_used].iterrows():
            result.append({"val": s})
            
        return result, similar_score

## Model Loading and Fine-Tuning Utilities

In [ ]:
from huggingface_hub import HfApi, create_repo
import shutil

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
max_seq_length = 8192

# Model loading and memory management utility
def load_model(model_name: str, load_in_4bit=True):
    """
    Load a model with memory optimization
    """
    # Load model with unsloth for reduced memory usage
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        dtype=None,  # Auto detection: Float16 for Tesla T4/V100, Bfloat16 for Ampere+
        load_in_4bit=load_in_4bit,  # 4-bit quantization
    )
    
    # Set template for some models
    if not hasattr(tokenizer, "chat_template") and tokenizer.chat_template is None and 'qwen' in model_name.lower():
        tokenizer.chat_template = "<|im_start|>system\n{{ messages[0]['content'] }}<|im_end|>\n{% for message in messages[1:] %}<|im_start|>{{ message['role'] }}\n{{ message['content'] }}<|im_end|>\n{% endfor %}"
        print("Qwen chat template has been set.")
        
    return model, tokenizer

def get_pipeline(model, tokenizer, max_new_tokens=512):
    """
    Create a pipeline from a model and tokenizer
    """
    pipe = pipeline(
        "text-generation",
        model=model, 
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        top_k=None,
        top_p=None,
        temperature=None,
        torch_dtype=torch.bfloat16 if is_bfloat16_supported() else torch.float16,
        return_full_text=False
    )
    
    return HuggingFacePipeline(pipeline=pipe)

## Prepare Datasets

In [ ]:
# Load test data
def load_test_data(path_to_data):
    with open(path_to_data, "r", encoding="utf-8") as file:
        raw = json.load(file)
    
    data = []
    for item in raw:
        data.append({
            "question": item['question'],
            "sparql": item.get('sparql', ''),
        })
    
    return pd.DataFrame(data)

# Load all datasets
test_df = load_test_data("test_data.json")

# Display information
print(f"Loaded {len(test_df)} test questions")

## Prepare Training Data for Fine-Tuning

In [ ]:
def check_template(model_name: str):
    """
    Load a model to get tokenizer and check if it has a chat template
    Returns both the boolean flag and the tokenizer for reuse
    """
    print(f"Loading tokenizer and checking chat template for {model_name}...")
    
    # Load model and tokenizer
    model, tokenizer = load_model(model_name)
    
    # Check if the tokenizer has a chat template
    has_template = hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None
    print(f"Model has chat template: {has_template}")
    
    # Clean up model but keep tokenizer
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    return has_template, tokenizer

# Set the global variable before dataset preparation
has_chat_template, tokenizer = check_template(model_name)
print(f"Global has_chat_template set to: {has_chat_template}")

## Fine-Tune Models

In [ ]:
import torch
import psutil
import gc

def print_gpu_memory():
    """Display detailed GPU memory information"""
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            total_memory = torch.cuda.get_device_properties(i).total_memory / 1024**3  # GB
            allocated_memory = torch.cuda.memory_allocated(i) / 1024**3  # GB
            reserved_memory = torch.cuda.memory_reserved(i) / 1024**3  # GB
            free_memory = total_memory - reserved_memory
            print(f"GPU {i}: Total: {total_memory:.2f} GB | Allocated: {allocated_memory:.2f} GB | Reserved: {reserved_memory:.2f} GB | Free: {free_memory:.2f} GB")
    else:
        print("No GPU available")

def print_system_memory():
    """Display system RAM information"""
    mem = psutil.virtual_memory()
    total_memory = mem.total / 1024**3  # GB
    available_memory = mem.available / 1024**3  # GB
    used_memory = mem.used / 1024**3  # GB
    percent_used = mem.percent
    
    print(f"System Memory: Total: {total_memory:.2f} GB | Used: {used_memory:.2f} GB ({percent_used}%) | Available: {available_memory:.2f} GB")

def print_all_memory():
    """Print both GPU and system memory information"""
    print("\n--- MEMORY STATUS ---")
    print_gpu_memory()
    print_system_memory()
    print("--------------------\n")

## Property Retrieval Implementation

In [ ]:
!pip install --upgrade -q protobuf

In [ ]:
# Start Local Weaviate
import weaviate
from weaviate.embedded import EmbeddedOptions

# Configure embedded options
embedded_options = EmbeddedOptions()

# Initialize the Weaviate client with embedded options
weaviate_client = weaviate.WeaviateClient(embedded_options=embedded_options)
weaviate_client.connect()

print(weaviate_client.is_ready())

In [ ]:
import weaviate
import weaviate.classes as wvc

class WikidataPropertyRetrieval:
    def __init__(
        self,
        df_properties: pd.DataFrame,
        embedding_model_name: str = "jinaai/jina-embeddings-v3",
    ) -> None:
        self.df_properties = df_properties
        self.model_embed = SentenceTransformer(embedding_model_name, trust_remote_code=True)
        self.stopwords = set(stopwords.words("english"))
        
        # Connect to local Weaviate
        self.client = weaviate_client
        
        # Create/get collection
        db_collection_name = "wikidata_property_db"
        if not self.client.collections.exists(db_collection_name):
            self.collection = self.client.collections.create(
                name=db_collection_name,
                vectorizer_config=wvc.config.Configure.Vectorizer.none(),
            )
            self.is_collection_empty = True
        else:
            self.collection = self.client.collections.get(db_collection_name)
            self.is_collection_empty = False
            
        # Initialize collection with data if empty
        if self.is_collection_empty:
            emb_properties = self.model_embed.encode(
                self.df_properties["label"].tolist(), show_progress_bar=True
            )

            with self.collection.batch.dynamic() as batch:
                for i, row in df_properties.iterrows():
                    batch.add_object(
                        properties=row.to_dict(),
                        vector=emb_properties[i].tolist(),
                    )

    def _search(self, q: str, k: int = 5) -> pd.DataFrame:
        query_vector = self.model_embed.encode([q])[0]
        response = self.collection.query.hybrid(
            query=q,
            query_properties=["label"],
            vector=query_vector,
            return_metadata=wvc.query.MetadataQuery(score=True),
            limit=k,
        )
        df = pd.DataFrame(
            [{**o.properties, "score": o.metadata.score} for o in response.objects]
        )
        return df

    def _preprocess_into_tokens(self, q: str) -> list[str]:
        tokenizer = RegexpTokenizer(r"\w+")
        tokenized = tokenizer.tokenize(q)
        return [tok.lower() for tok in tokenized if tok.lower() not in self.stopwords]

    def _generate_ngrams(self, tokens: list[str]) -> list[str]:
        max_n = min(len(tokens), 3)
        result = []
        for n in range(1, max_n + 1):
            n_grams = ngrams(tokens, n)
            result.extend([" ".join(ng) for ng in n_grams])
        return result

    def get_related_candidates(
        self,
        q: str,
        property_candidates: list[str] = [],
        threshold: float = 0.5,
        k: int = 5,
    ) -> dict[str, list[str]]:
        tokens = self._preprocess_into_tokens(q)
        ngrams = self._generate_ngrams(tokens)
        result = {"properties": []}

        for ngram in ngrams + property_candidates:
            df_res = self._search(ngram, k=k)
            if not df_res.empty:
                df_res["idWithLabel"] = df_res["propertyId"] + " - " + df_res["label"]
                filtered_results = df_res[df_res["score"] >= threshold]["idWithLabel"].tolist()
                if filtered_results:
                    result["properties"].extend(filtered_results)
                    result["properties"] = list(set(result["properties"]))

        result["properties"] = sorted(result["properties"])
        return result

## Extraction Functions

In [ ]:
prop_retrieval = WikidataPropertyRetrieval(
    pd.read_csv("./data/wikidata_ontology/properties.csv"),
    embedding_model_name="jinaai/jina-embeddings-v3",
)

# Initialize verbalization
verbalization = WikidataVerbalization(
    model_name="jinaai/jina-embeddings-v3",
    query_model_encode_kwargs={
        "task": "retrieval.query",
        "prompt_name": "retrieval.query",
    },
    passage_model_encode_kwargs={
        "task": "retrieval.passage",
        "prompt_name": "retrieval.passage",
    },
)

In [ ]:
# Entity and property extraction function (works for both base and fine-tuned models)
def extract_entities_and_properties(questions, model_path):
    """
    Extract entities and properties from questions using specified model
    """
    # Load model
    model, tokenizer = load_model(model_path)
    pipe = get_pipeline(model, tokenizer)

    system_prompt = """You are an expert entity and property extractor for knowledge graph querying. Your task is to analyze a natural language question and identify the relevant entities and properties needed to create a SPARQL query for Wikidata.

Guidelines:
1. For each question, extract ALL entities mentioned in the question
2. For each question, extract ALL relevant properties needed to answer the question
3. Format your response as a structured JSON object with 'entities' and 'properties' keys
4. Each key should contain an array of strings with the entity or property names
5. Focus ONLY on extracting, not on generating SPARQL queries

Your output should look like:
```json
{
  "entities": ["entity1", "entity2", ...],
  "properties": ["property1", "property2", ...]
}
```
"""

    results = []
    for question in tqdm(questions, desc=f"Extracting entities and properties ({model_path})"):
        user_prompt = f"""Question: {question}

Extract all entities and properties from this question that would be needed to generate a SPARQL query for Wikidata."""

        if has_chat_template:
            prompt = tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
        else:    
            prompt = system_prompt + '\n' + user_prompt + '\n'

        # Generate response
        response = pipe.invoke(prompt)
        
        # Extract JSON from response
        match = re.search(r"```(?:json)?\s*([\s\S]*?)```", response)
        if match:
            print("gansik")
            json_str = match.group(1).strip()
            try:
                extracted = json.loads(json_str)
                print(extracted)
                results.append({
                    "question": question,
                    "entities": extracted.get("entities", []),
                    "properties": extracted.get("properties", [])
                })
            except json.JSONDecodeError:
                print("json decode error")
                results.append({
                    "question": question,
                    "entities": [],
                    "properties": []
                })
        else:
            print("ga match")
            results.append({
                "question": question,
                "entities": [],
                "properties": []
            })
    
    # Clean up memory
    del model, tokenizer, pipe
    gc.collect()
    torch.cuda.empty_cache()
    
    return pd.DataFrame(results)

In [ ]:
def get_entity_uris(extraction_df):
    """
    Get the most appropriate entity URIs for extracted entities
    """
    # Load base model
    model, tokenizer = load_model(model_name)
    pipe = get_pipeline(model, tokenizer)

    # Create chat model for langchain
    chat_model = ChatHuggingFace(llm=pipe)

    api = WikidataAPI()
    result_df = extraction_df.copy()
    result_df['entity_uris'] = [[] for _ in range(len(result_df))]

    for idx, row in tqdm(result_df.iterrows(), total=len(result_df), desc="Getting entity URIs"):
        entity_uris = []

        for entity in row['entities']:
            retrieved_entities, error = api.get_entities(entity, k=5)

            if error or not retrieved_entities:
                continue

            # Define Pydantic model for output parsing
            class Entity(BaseModel):
                """Represents the most appropriate Wikidata entity ID"""
                id: str = Field(..., description="Wikidata Entity ID")

            output_parser = PydanticOutputParser(pydantic_object=Entity)
            format_instructions = output_parser.get_format_instructions()

            chat_prompt_template = ChatPromptTemplate.from_messages(
                [
                    (
                        "system",
                        """Find the most appropriate Wikidata entity ID for the given entity to answer the question.
Only return the entity ID from the provided list. Do not include explanations.
{format_instructions}""",
                    ),
                    MessagesPlaceholder("chat_history"),
                    (
                        "human",
                        """Retrieved entities:
{retrieved_entities}
Question: {question}
Entity: {input}
Entity ID:""",
                    ),
                ]
            )

            final_prompt = chat_prompt_template.partial(
                format_instructions=format_instructions,
                retrieved_entities=retrieved_entities,
                question=row['question'],
            )

            try:
                llm_chain = final_prompt | chat_model | StrOutputParser()
                completion = llm_chain.invoke({"chat_history": [], "input": entity})
                entity = output_parser.parse(completion)
                entity_uris.append(entity.id)
            except Exception as e:
                print(f"Error identifying entity URI: {e}")

        result_df.at[idx, 'entity_uris'] = entity_uris

    # Clean up memory
    del model, tokenizer, pipe, chat_model
    gc.collect()
    torch.cuda.empty_cache()

    return result_df

In [ ]:
# SPARQL generation function with verbalization threshold and retry logic
def generate_sparql(extraction_df, model_path, try_threshold=5):
    """
    Generate SPARQL queries using specified model
    Only generates SPARQL if verbalization score < 0.6
    Retries up to try_threshold times if SPARQL execution fails
    """
    # Load model
    model, tokenizer = load_model(model_path)
    pipe = get_pipeline(model, tokenizer, max_new_tokens=1024)
    
    api = WikidataAPI()
    result_df = extraction_df.copy()
    result_df['generated_sparql'] = ["" for _ in range(len(result_df))]
    result_df['sparql_results'] = [[] for _ in range(len(result_df))]
    result_df['sparql_generation_attempts'] = [0 for _ in range(len(result_df))]
    result_df['used_verbalization'] = [False for _ in range(len(result_df))]
    
    system_prompt = """You are a SPARQL generator expert for Wikidata knowledge graph. Your task is to convert the following natural language question to a SPARQL query for Wikidata using the provided entity and property resolutions.

Guidelines:
1. First identify which entities from the list match the question's intent
2. Identify which entities are relevant to the question and select EXACTLY ONE entity ID for each distinct concept in the question
3. When multiple entities have similar labels, choose the one whose description best matches the question's context
4. From the properties list, choose which properties are needed to answer the question
5. Select only the minimum necessary properties required to answer the question correctly
6. Use ALL identified entities and necessary properties in your SPARQL query
7. Use PREFIX NOTATION ONLY (e.g., wd:Q123, wdt:P123), NOT full URIs
8. Optimize your query by using appropriate SPARQL features (DISTINCT, FILTER, ORDER BY, LIMIT) when needed
9. Return entity IDs directly without using label services
10. Return the raw SPARQL query in this format:
   ```sparql
   <your_sparql_query_here>
   ```

IMPORTANT: Before generating the SPARQL query, provide your step-by-step reasoning about how to construct the query.
"""
    
    for idx, row in tqdm(result_df.iterrows(), total=len(result_df), desc=f"Generating SPARQL queries ({model_path})"):
        # Check verbalization score first
        verbalization_score = row.get('verbalization_scores', 0.0)
        
        if verbalization_score >= 0.6:
            # Use verbalization results, skip SPARQL generation
            result_df.at[idx, 'used_verbalization'] = True
            print(f"Question {idx}: Using verbalization (score: {verbalization_score:.3f})")
            continue
        
        # Verbalization score < 0.6, proceed with SPARQL generation
        print(f"Question {idx}: Generating SPARQL (verbalization score: {verbalization_score:.3f})")
        
        # Format entities matches
        entities_matches = {}
        for entity_uri in row['entity_uris']:
            entities, _ = api.get_entities(entity_uri, k=1)
            if entities:
                entities_matches[entity_uri] = entities
        
        properties_candidates = prop_retrieval.get_related_candidates(
            row['question'], 
            property_candidates=row['properties'],
            threshold=0.6
        )
        
        # Format properties matches
        properties_matches = {}
        for prop in properties_candidates['properties']:
            prop_id = prop.split(' - ')[0]
            prop_label = prop.split(' - ')[1]
            if prop_id not in properties_matches:
                properties_matches[prop_id] = []
            properties_matches[prop_id].append({
                'id': prop_id,
                'label': prop_label,
                'description': ""
            })
        
        # Helper functions for formatting
        def format_entity_matches(matches):
            result = ""
            for item, match_list in matches.items():
                for match in match_list:
                    result += f"- id: {match['uri']}, label: {match['label']}, description: {match.get('description', '')}\n"
            return result

        def format_property_matches(matches):
            result = ""
            for item, match_list in matches.items():
                for match in match_list:
                    result += f"- id: {match['id']}, label: {match['label']}, description: {match.get('description', '')}\n"
            return result
            
        entities_matches_formatted = format_entity_matches(entities_matches)
        properties_matches_formatted = format_property_matches(properties_matches)
        
        user_prompt = f"""Question: {row['question']}

Entities:
{entities_matches_formatted}

Properties:
{properties_matches_formatted}

SPARQL:"""

        # Try generating SPARQL up to try_threshold times
        attempts = 0
        successful = False
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        while attempts < try_threshold and not successful:
            attempts += 1
            
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

            # Generate response
            response = pipe.invoke(prompt)
            
            # Extract SPARQL from response
            match = re.search(r"```(?:sparql)?\s*([\s\S]*?)```", response)
            if match:
                sparql = match.group(1).strip()
                result_df.at[idx, 'generated_sparql'] = sparql
                
                # Execute SPARQL query
                results, error = api.execute_sparql(sparql)
                if not error and results:
                    result_df.at[idx, 'sparql_results'] = results
                    successful = True
                    print(f"  Attempt {attempts}: Success")
                else:
                    print(f"  Attempt {attempts}: Failed - {error if error else 'Empty results'}")
                    messages.append(
                        {"role": "assistant", "content": response}
                    )
                    messages.append(
                        {"role": "user", "content": f"""The SPARQL query you generated {"returned an error" if error else "produced empty results"}. 
Please generate a better query using different properties, entities, or query structure to address the original question."""}
                    )
            else:
                print(f"  Attempt {attempts}: No SPARQL found in response")
                messages.append(
                    {"role": "assistant", "content": response}
                )
                messages.append(
                    {"role": "user", "content": """I need a SPARQL query to answer my question, but no valid SPARQL code was found in your response.
Please provide a complete, executable SPARQL query enclosed in triple backticks (```).```"""}
                )

        
        result_df.at[idx, 'sparql_generation_attempts'] = attempts
    
    # Clean up memory
    del model, tokenizer, pipe
    gc.collect()
    torch.cuda.empty_cache()
    
    return result_df

In [ ]:
# Verbalization function (using original logic)
def verbalize_entities(extraction_df):
    """
    Run verbalization for the identified entities
    """
    result_df = extraction_df.copy()
    result_df['verbalization_results'] = [[] for _ in range(len(result_df))]
    result_df['verbalization_scores'] = [0.0 for _ in range(len(result_df))]
    
    for idx, row in tqdm(result_df.iterrows(), total=len(result_df), desc="Verbalizing entities"):
        best_result = []
        best_score = 0.0
        
        for entity_uri in row['entity_uris']:
            try:
                result, score = verbalization.run(row['question'], entity_uri)
                if score > best_score:
                    best_result = result
                    best_score = score
            except Exception as e:
                print(f"Error in verbalization: {e}")
        
        result_df.at[idx, 'verbalization_results'] = best_result
        result_df.at[idx, 'verbalization_scores'] = best_score
    
    return result_df

## Main Evaluation Pipeline

In [ ]:
def compare_two_dataframes(df1: pd.DataFrame, df2: pd.DataFrame) -> Dict[str, float]:
    # df1: DataFrame for ground truth
    # df2: DataFrame for predicted
    if len(df1.columns) != len(df2.columns):
        return {
                    'jaccard': 0,
                    'recall': 0,
                    'precision': 0,
                    'f1': 0,
                    'tp': 0,
                    'fp': 0,
                    'fn': 0,
                    'tn': 0
                }

    set1, set2 = set(), set()
    for _, row in df1.iterrows():
        row = list(row)
        row = sorted(row)
        row = tuple(row)
        set1.add(row)

    for _, row in df2.iterrows():
        row = list(row)
        row = sorted(row)
        row = tuple(row)
        set2.add(row)
    
    jaccard = len(set1 & set2) / len(set1 | set2) if len(set1 | set2) > 0 else 0
    # recall = correct retrieved / all ground truth
    recall = len(set1 & set2) / len(set1) if len(set1) > 0 else 0
    # precision = correct retrieved / retrieved answers
    precision = len(set1 & set2) / len(set2) if len(set2) > 0 else 0
    # f1 score = 2 x prec x recall / (prec + recall)
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # TP, TN, FP, FN computation (might be useful for computing micro metrics)
    tp = len(set1 & set2)
    fp = len(set2) - tp
    fn = len(set1) - tp
    total_pairs = len(set1) + len(set2) - tp
    tn = total_pairs - (tp + fp + fn)

    return {
        'jaccard': jaccard,
        'recall': recall,
        'precision': precision,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn
    }

In [ ]:
def evaluate_batch_results(results_df, ground_truth_queries, questions):
    """
    Evaluate the batch results from the FROG pipeline
    """
    api = WikidataAPI()
    evaluation_results = []
    
    print(f"Evaluating {len(results_df)} questions...")
    
    for idx, row in results_df.iterrows():
        question = row['question']
        generated_query = row.get('generated_sparql', '')
        sparql_results = row.get('sparql_results', [])
        verbalization_results = row.get("verbalization_results", [])
        verbalization_score = row.get('verbalization_scores', 0.0)
        used_verbalization = row.get('used_verbalization', False)
        ground_truth_query = ground_truth_queries[idx]
        
        print(f"\n=== Question {idx + 1}: {question} ===")
        
        result_entry = {
            "question_id": idx,
            "question_text": question,
            "ground_truth_query": ground_truth_query,
            "generated_query": generated_query,
            "entities": row.get('entities', []),
            "properties": row.get('properties', []),
            "entity_uris": row.get('entity_uris', []),
            "verbalization_score": verbalization_score,
            "used_verbalization": used_verbalization,
            "sparql_generation_attempts": row.get('sparql_generation_attempts', 0),
            "approach_used": "verbalization" if used_verbalization else "sparql_generation",
            "metrics": {},
            "error": None
        }
        
        try:
            # Get ground truth results
            ground_truth_df = api.execute_sparql_to_df(ground_truth_query)
            print(f"Ground truth shape: {ground_truth_df.shape}")
            print(ground_truth_df.head())

            # Determine which results to use
            if used_verbalization and verbalization_results:
                # Use verbalization results
                pred_df = pd.DataFrame(verbalization_results)
                print(f"Using verbalization results (score: {verbalization_score:.3f})")
            elif sparql_results:
                # Use SPARQL results
                pred_df = pd.DataFrame(sparql_results)
                print(f"Using SPARQL results (attempts: {row.get('sparql_generation_attempts', 0)})")
            else:
                pred_df = pd.DataFrame()
                print("No results available")
            
            if not pred_df.empty:
                print(f"Prediction shape: {pred_df.shape}")
                print(pred_df.head())
                
                # Compare results
                metrics = compare_two_dataframes(ground_truth_df, pred_df)
                result_entry["metrics"] = metrics
                
                print(f"Metrics: Jaccard={metrics['jaccard']:.3f}, Precision={metrics['precision']:.3f}, Recall={metrics['recall']:.3f}")
            else:
                print("No results to evaluate")
                result_entry["metrics"] = {
                    'jaccard': 0, 'recall': 0, 'precision': 0, 'f1': 0,
                    'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0
                }
                result_entry["error"] = "No results generated"
                
        except Exception as e:
            print(f"Error evaluating question {idx}: {e}")
            result_entry["error"] = str(e)
            result_entry["metrics"] = {
                'jaccard': 0, 'recall': 0, 'precision': 0, 'f1': 0,
                'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0
            }
        
        evaluation_results.append(result_entry)
    
    return evaluation_results

In [ ]:
def calculate_summary_statistics(evaluation_results):
    """Calculate summary statistics from evaluation results"""    
    # Calculate average metrics - CHANGED: Include all results with metrics, even if they have errors
    valid_results = [r for r in evaluation_results if r["metrics"]]  # Removed: and not r["error"]
    
    if not valid_results:
        print("No valid results to summarize")
        return {}
    
    avg_metrics = {}
    for metric in valid_results[0]["metrics"].keys():
        avg_metrics[metric] = sum(r["metrics"][metric] for r in valid_results) / len(valid_results)
    
    # Calculate success rates
    total_questions = len(evaluation_results)
    successful_sparql = len([r for r in evaluation_results if r["generated_query"] and not r["error"]])
    successful_results = len([r for r in evaluation_results if r["metrics"]["f1"] > 0])
    
    # Verbalization vs SPARQL statistics
    verbalization_used = len([r for r in evaluation_results if r.get("used_verbalization", False)])
    sparql_attempts = [r.get("sparql_generation_attempts", 0) for r in evaluation_results if not r.get("used_verbalization", False)]
    avg_sparql_attempts = sum(sparql_attempts) / len(sparql_attempts) if sparql_attempts else 0
    
    # Entity and property extraction stats
    avg_entities = sum(len(r["entities"]) for r in evaluation_results) / total_questions
    avg_properties = sum(len(r["properties"]) for r in evaluation_results) / total_questions
    avg_entity_uris = sum(len(r["entity_uris"]) for r in evaluation_results) / total_questions
    avg_verbalization_score = sum(r["verbalization_score"] for r in evaluation_results) / total_questions
    
    summary = {
        "total_questions": total_questions,
        "successful_sparql_generation": successful_sparql,
        "successful_results": successful_results,
        "sparql_success_rate": successful_sparql / total_questions,
        "results_success_rate": successful_results / total_questions,
        "verbalization_used_count": verbalization_used,
        "verbalization_usage_rate": verbalization_used / total_questions,
        "avg_sparql_attempts": avg_sparql_attempts,
        "average_metrics": avg_metrics,
        "avg_entities_per_question": avg_entities,
        "avg_properties_per_question": avg_properties,
        "avg_entity_uris_per_question": avg_entity_uris,
        "avg_verbalization_score": avg_verbalization_score,
    }
    
    return summary

In [ ]:
# Unified evaluation pipeline
def evaluate_questions_pipeline(questions, entity_model_path, sparql_model_path):
    """
    Run the complete evaluation pipeline with specified models
    """
    # Step 1: Extract entities and properties
    print(f"Step 1: Extracting entities and properties using {entity_model_path}")
    entity_property_extraction = extract_entities_and_properties(questions, entity_model_path)
    
    # Step 2: Get entity URIs
    print(f"Step 2: Identifying entity URIs")
    entity_uri_df = get_entity_uris(entity_property_extraction)
    
    # Step 3: Run verbalization
    print(f"Step 3: Running verbalization")
    verbalization_df = verbalize_entities(entity_uri_df)
    
    # Step 4: Generate SPARQL (with verbalization threshold check)
    print(f"Step 4: Generating SPARQL queries using {sparql_model_path}")
    final_df = generate_sparql(verbalization_df, sparql_model_path, try_threshold=5)
    
    return final_df

## Perform Evaluation

In [ ]:
# Run comparative evaluation on test questions
questions = test_df['question'].tolist()
ground_truth_queries = test_df['sparql'].tolist()

print("Starting comparative FROG evaluation pipeline...")
print("=" * 60)

# Evaluate fine-tuned models
print("\n1. EVALUATING FINE-TUNED MODELS")
print("=" * 40)
# results_df_finetuned = evaluate_questions_pipeline(
#     questions, 
#     entity_model_path="/kaggle/input/ft-wikidata-qwen2-5-7b-i/ft_model_entity_prop (1)/ft_model_entity_prop",
#     sparql_model_path="/kaggle/input/ft-wikidata-qwen2-5-7b-i/ft_model_sparql (1)/ft_model_sparql",
# )

# Pindah ke sini biar bisa clear_memory()
entity_model_path="/kaggle/input/ft-wikidata-mistral-nemo-instruct/ft_model_entity_prop (3)/ft_model_entity_prop"
sparql_model_path="/kaggle/input/ft-wikidata-mistral-nemo-instruct/ft_model_sparql (3)/ft_model_sparql"

# Step 1: Extract entities and properties
print(f"Step 1: Extracting entities and properties using {entity_model_path}")
entity_property_extraction = extract_entities_and_properties(questions, entity_model_path)
clear_memory()
clear_memory()

# Step 2: Get entity URIs
print(f"Step 2: Identifying entity URIs")
entity_uri_df = get_entity_uris(entity_property_extraction)
clear_memory()
clear_memory()

# Step 3: Run verbalization
print(f"Step 3: Running verbalization")
verbalization_df = verbalize_entities(entity_uri_df)

# Step 4: Generate SPARQL (with verbalization threshold check)
print(f"Step 4: Generating SPARQL queries using {sparql_model_path}")
results_df_finetuned = generate_sparql(verbalization_df, sparql_model_path, try_threshold=5)
clear_memory()
clear_memory()

In [ ]:
# Evaluate base model
print("\n2. EVALUATING BASE MODEL")
print("=" * 40) 
# results_df_base = evaluate_questions_pipeline(
#     questions,
#     entity_model_path=model_name,
#     sparql_model_path=model_name,
# )

# Pindah ke sini biar bisa clear_memory()
entity_model_path=model_name
sparql_model_path=model_name

# Step 1: Extract entities and properties
print(f"Step 1: Extracting entities and properties using {entity_model_path}")
entity_property_extraction = extract_entities_and_properties(questions, entity_model_path)
clear_memory()
clear_memory()

# Step 2: Get entity URIs
print(f"Step 2: Identifying entity URIs")
entity_uri_df = get_entity_uris(entity_property_extraction)
clear_memory()
clear_memory()

# Step 3: Run verbalization
print(f"Step 3: Running verbalization")
verbalization_df = verbalize_entities(entity_uri_df)

# Step 4: Generate SPARQL (with verbalization threshold check)
print(f"Step 4: Generating SPARQL queries using {sparql_model_path}")
results_df_base = generate_sparql(verbalization_df, sparql_model_path, try_threshold=5)
clear_memory()
clear_memory()

In [ ]:
print("\n3. DETAILED EVALUATION OF RESULTS")
print("=" * 40)

# Evaluate fine-tuned model results
print("Evaluating fine-tuned model results...")
evaluation_results_finetuned = evaluate_batch_results(
    results_df_finetuned, 
    ground_truth_queries, 
    questions
)

# Calculate summary statistics for fine-tuned model
summary_stats_finetuned = calculate_summary_statistics(evaluation_results_finetuned)

print("\nEvaluating base model results...")
# Evaluate base model results
evaluation_results_base = evaluate_batch_results(
    results_df_base, 
    ground_truth_queries, 
    questions
)

# Calculate summary statistics for base model
summary_stats_base = calculate_summary_statistics(evaluation_results_base)

In [ ]:
print("\n4. COMPARATIVE ANALYSIS")
print("=" * 40)

# Display comprehensive comparative results
print("=" * 80)
print("FROG COMPARATIVE EVALUATION RESULTS")
print("=" * 80)

def display_model_results(title, summary_stats):
    print(f"\n{title}")
    print("=" * len(title))
    
    print(f"Total Questions Processed: {summary_stats['total_questions']}")
    print(f"SPARQL Generation Success Rate: {summary_stats['sparql_success_rate']:.1%}")
    print(f"Results Success Rate: {summary_stats['results_success_rate']:.1%}")
    
    print("\nPipeline Component Averages:")
    print(f"  Entities per question: {summary_stats['avg_entities_per_question']:.2f}")
    print(f"  Properties per question: {summary_stats['avg_properties_per_question']:.2f}")
    print(f"  Entity URIs per question: {summary_stats['avg_entity_uris_per_question']:.2f}")
    print(f"  Verbalization score: {summary_stats['avg_verbalization_score']:.4f}")
    print(f"  Verbalization usage rate: {summary_stats['verbalization_usage_rate']:.1%}")
    print(f"  Average SPARQL attempts: {summary_stats['avg_sparql_attempts']:.2f}")
    
    if summary_stats.get("average_metrics"):
        print("\nAverage Evaluation Metrics:")
        for metric, score in summary_stats["average_metrics"].items():
            print(f"  {metric.upper()}: {score:.4f}")

# Display results for both models
display_model_results("FINE-TUNED MODEL RESULTS", summary_stats_finetuned)
display_model_results("BASE MODEL RESULTS", summary_stats_base)

# Comparative analysis
print("\n" + "=" * 40)
print("COMPARATIVE ANALYSIS")
print("=" * 40)

print(f"\nSPARQL Generation Success Rate:")
print(f"  Fine-tuned: {summary_stats_finetuned['sparql_success_rate']:.1%}")
print(f"  Base Model: {summary_stats_base['sparql_success_rate']:.1%}")
improvement_sparql = summary_stats_finetuned['sparql_success_rate'] - summary_stats_base['sparql_success_rate']
print(f"  Improvement: {improvement_sparql:+.1%}")

print(f"\nResults Success Rate:")
print(f"  Fine-tuned: {summary_stats_finetuned['results_success_rate']:.1%}")
print(f"  Base Model: {summary_stats_base['results_success_rate']:.1%}")
improvement_results = summary_stats_finetuned['results_success_rate'] - summary_stats_base['results_success_rate']
print(f"  Improvement: {improvement_results:+.1%}")

print(f"\nVerbalization Usage:")
print(f"  Fine-tuned: {summary_stats_finetuned['verbalization_usage_rate']:.1%}")
print(f"  Base Model: {summary_stats_base['verbalization_usage_rate']:.1%}")

if summary_stats_finetuned.get("average_metrics") and summary_stats_base.get("average_metrics"):
    print(f"\nAverage F1 Score:")
    f1_finetuned = summary_stats_finetuned["average_metrics"]["f1"]
    f1_base = summary_stats_base["average_metrics"]["f1"]
    print(f"  Fine-tuned: {f1_finetuned:.4f}")
    print(f"  Base Model: {f1_base:.4f}")
    print(f"  Improvement: {(f1_finetuned - f1_base):+.4f}")
    
    print(f"\nAverage Precision:")
    prec_finetuned = summary_stats_finetuned["average_metrics"]["precision"]
    prec_base = summary_stats_base["average_metrics"]["precision"]
    print(f"  Fine-tuned: {prec_finetuned:.4f}")
    print(f"  Base Model: {prec_base:.4f}")
    print(f"  Improvement: {(prec_finetuned - prec_base):+.4f}")
    
    print(f"\nAverage Recall:")
    recall_finetuned = summary_stats_finetuned["average_metrics"]["recall"]
    recall_base = summary_stats_base["average_metrics"]["recall"]
    print(f"  Fine-tuned: {recall_finetuned:.4f}")
    print(f"  Base Model: {recall_base:.4f}")
    print(f"  Improvement: {(recall_finetuned - recall_base):+.4f}")
    
    print(f"\nAverage Jaccard Similarity:")
    jaccard_finetuned = summary_stats_finetuned["average_metrics"]["jaccard"]
    jaccard_base = summary_stats_base["average_metrics"]["jaccard"]
    print(f"  Fine-tuned: {jaccard_finetuned:.4f}")
    print(f"  Base Model: {jaccard_base:.4f}")
    print(f"  Improvement: {(jaccard_finetuned - jaccard_base):+.4f}")

In [ ]:
# Save detailed comparative results
print("\n5. SAVING RESULTS")
print("=" * 20)

comparative_results = {
    "evaluation_metadata": {
        "total_questions": len(questions),
        "model_info": {
            "base_model_name": model_name,
            "finetuned_entity_model_path": "ft_entity_prop_model",
            "finetuned_sparql_model_path": "ft_sparql_model"
        },
        "evaluation_date": "2024-05-24"
    },
    "finetuned_model": {
        "evaluation_results": evaluation_results_finetuned,
        "summary_statistics": summary_stats_finetuned,
        "pipeline_results": results_df_finetuned.to_dict('records')
    },
    "base_model": {
        "evaluation_results": evaluation_results_base,
        "summary_statistics": summary_stats_base,
        "pipeline_results": results_df_base.to_dict('records')
    },
    "comparative_improvements": {
        "sparql_success_rate_improvement": improvement_sparql,
        "results_success_rate_improvement": improvement_results,
        "metric_improvements": {}
    }
}

# Add metric improvements if available
if summary_stats_finetuned.get("average_metrics") and summary_stats_base.get("average_metrics"):
    for metric in summary_stats_finetuned["average_metrics"]:
        improvement = summary_stats_finetuned["average_metrics"][metric] - summary_stats_base["average_metrics"][metric]
        comparative_results["comparative_improvements"]["metric_improvements"][metric] = improvement

# Save results to JSON files
with open('frog_comparative_evaluation_results.json', 'w') as f:
    json.dump(comparative_results, f, indent=2)

# Save individual model results
results_df_finetuned.to_json('frog_finetuned_pipeline_results.json', orient='records', indent=2)
results_df_base.to_json('frog_base_pipeline_results.json', orient='records', indent=2)

# Save detailed evaluation results separately
with open('frog_finetuned_detailed_evaluation.json', 'w') as f:
    json.dump(evaluation_results_finetuned, f, indent=2)

with open('frog_base_detailed_evaluation.json', 'w') as f:
    json.dump(evaluation_results_base, f, indent=2)

print("Results saved to:")
print("- 'frog_comparative_evaluation_results.json' (Complete comparative analysis)")
print("- 'frog_finetuned_pipeline_results.json' (Fine-tuned pipeline results)")
print("- 'frog_base_pipeline_results.json' (Base model pipeline results)")
print("- 'frog_finetuned_detailed_evaluation.json' (Fine-tuned detailed evaluation)")
print("- 'frog_base_detailed_evaluation.json' (Base model detailed evaluation)")

# Display examples of successful and failed queries
def show_examples(evaluation_results, model_type):
    print(f"\n--- {model_type} Examples ---")
    
    # Find successful example
    successful = [r for r in evaluation_results if r["metrics"]["f1"] > 0]
    if successful:
        example = successful[0]
        print(f"\n✓ SUCCESSFUL EXAMPLE:")
        print(f"Question: {example['question_text']}")
        print(f"Approach: {example['approach_used']}")
        print(f"Generated Query: {example['generated_query'][:100]}...")
        print(f"F1 Score: {example['metrics']['f1']:.4f}")
    
    # Find failed example
    failed = [r for r in evaluation_results if r["metrics"]["f1"] == 0]
    if failed:
        example = failed[0]
        print(f"\n✗ FAILED EXAMPLE:")
        print(f"Question: {example['question_text']}")
        print(f"Approach: {example['approach_used']}")
        print(f"Error: {example.get('error', 'No results generated')}")
        if example['generated_query']:
            print(f"Generated Query: {example['generated_query'][:100]}...")

show_examples(evaluation_results_finetuned, "FINE-TUNED MODEL")
show_examples(evaluation_results_base, "BASE MODEL")

print("\n" + "=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)